<a href="https://colab.research.google.com/github/javageek2018/AirlineArrivalDelay/blob/%E2%80%9Cflight_data%E2%80%9D/Flights_Angela.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
from pyspark.sql.functions import col

# Connect to GitHub

In [ ]:
!git clone https://github.com/javageek2018/AirlineArrivalDelay.git

Cloning into 'AirlineArrivalDelay'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 50 (delta 17), reused 45 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 4.51 MiB | 25.51 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [ ]:
%cd AirlineArrivalDelay

/content/AirlineArrivalDelay


In [ ]:
!git fetch --all

Fetching origin


In [ ]:
!git branch -a

* main
  remotes/origin/EDA
  remotes/origin/HEAD -> origin/main
  remotes/origin/main
  remotes/origin/“flight_data”


# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Set Up Spark

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jdk-headless is already the newest version (11.0.30+7-1ubuntu1~22.04).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.


In [ ]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [ ]:
import pyspark
print(pyspark.__version__)

4.0.2


In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ColabSpark") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

spark

# Read Data (Spark)

In [ ]:
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

In [ ]:
# file_path = "/content/drive/MyDrive/OMDS Capstone/Data/bts_with_weather.parquet"
file_path = "/content/drive/MyDrive/OMDS Capstone/Data/bts_with_weather_holiday.parquet"

df = spark.read.parquet(file_path)

In [ ]:
df.head()

Row(Year=2018, Quarter=1, Month=1, DayofMonth=1, DayOfWeek=1, FlightDate=1514764800000000000, Reporting_Airline='WN', Flight_Number_Reporting_Airline='1491.0', Origin='ABQ', Dest='BWI', CRSDepTime=730, DepTimeBlk='0700-0759', CRSArrTime=1310, ArrDel15=0, CRSElapsedTime=220.0, Distance=1670.0, DistanceGroup=7, date='2018-01-01', dep_hour=7, arr_hour=13, dep_hour_minus2=5, arr_hour_minus2=11, origin_temp_f=21.0, origin_dewpoint_f=8.1, origin_humidity=56.88, origin_feels_like_f=9.17, origin_wind_kts=8.818181818181818, origin_gust_kts=None, origin_visibility=10.0, origin_precip_in=0.0, origin_wx_codes='', origin_is_rain=0.0, origin_is_snow=0.0, origin_is_fog=0.0, origin_low_visibility=0.0, origin_high_wind=0.0, origin_severe_weather=0.0, dest_temp_f=21.9, dest_dewpoint_f=3.0, dest_humidity=43.38, dest_feels_like_f=9.7, dest_wind_kts=9.692307692307692, dest_gust_kts=16.0, dest_visibility=10.0, dest_precip_in=0.0, dest_wx_codes='', dest_is_rain=0.0, dest_is_snow=0.0, dest_is_fog=0.0, dest_lo

In [ ]:
df.dtypes

[('Year', 'bigint'),
 ('Quarter', 'bigint'),
 ('Month', 'bigint'),
 ('DayofMonth', 'bigint'),
 ('DayOfWeek', 'bigint'),
 ('FlightDate', 'bigint'),
 ('Reporting_Airline', 'string'),
 ('Flight_Number_Reporting_Airline', 'string'),
 ('Origin', 'string'),
 ('Dest', 'string'),
 ('CRSDepTime', 'bigint'),
 ('DepTimeBlk', 'string'),
 ('CRSArrTime', 'bigint'),
 ('ArrDel15', 'bigint'),
 ('CRSElapsedTime', 'double'),
 ('Distance', 'double'),
 ('DistanceGroup', 'bigint'),
 ('date', 'string'),
 ('dep_hour', 'bigint'),
 ('arr_hour', 'bigint'),
 ('dep_hour_minus2', 'bigint'),
 ('arr_hour_minus2', 'bigint'),
 ('origin_temp_f', 'double'),
 ('origin_dewpoint_f', 'double'),
 ('origin_humidity', 'double'),
 ('origin_feels_like_f', 'double'),
 ('origin_wind_kts', 'double'),
 ('origin_gust_kts', 'double'),
 ('origin_visibility', 'double'),
 ('origin_precip_in', 'double'),
 ('origin_wx_codes', 'string'),
 ('origin_is_rain', 'double'),
 ('origin_is_snow', 'double'),
 ('origin_is_fog', 'double'),
 ('origin_low

In [ ]:
df.createOrReplaceTempView("flights")

spark.sql("""
    SELECT DISTINCT origin_severe_weather
    FROM flights
""").show()

+---------------------+
|origin_severe_weather|
+---------------------+
|                  0.0|
|                  1.0|
|                 NULL|
+---------------------+



In [ ]:
# convert binary columns to int

double_cols = [
    'origin_is_rain',
    'origin_is_snow',
    'origin_is_fog',
    'origin_low_visibility',
    'origin_high_wind',
    'origin_severe_weather',
    'dest_is_rain',
    'dest_is_snow',
    'dest_is_fog',
    'dest_low_visibility',
    'dest_high_wind',
    'dest_severe_weather'
]

double_cols

['origin_is_rain',
 'origin_is_snow',
 'origin_is_fog',
 'origin_low_visibility',
 'origin_high_wind',
 'origin_severe_weather',
 'dest_is_rain',
 'dest_is_snow',
 'dest_is_fog',
 'dest_low_visibility',
 'dest_high_wind',
 'dest_severe_weather']

In [ ]:
df = df.withColumn("origin_is_rain",        col("origin_is_rain").cast("int"))
df = df.withColumn("origin_is_snow",        col("origin_is_snow").cast("int"))
df = df.withColumn("origin_is_fog",         col("origin_is_fog").cast("int"))
df = df.withColumn("origin_low_visibility", col("origin_low_visibility").cast("int"))
df = df.withColumn("origin_high_wind",      col("origin_high_wind").cast("int"))
df = df.withColumn("origin_severe_weather", col("origin_severe_weather").cast("int"))

df = df.withColumn("dest_is_rain",          col("dest_is_rain").cast("int"))
df = df.withColumn("dest_is_snow",          col("dest_is_snow").cast("int"))
df = df.withColumn("dest_is_fog",           col("dest_is_fog").cast("int"))
df = df.withColumn("dest_low_visibility",   col("dest_low_visibility").cast("int"))
df = df.withColumn("dest_high_wind",        col("dest_high_wind").cast("int"))
df = df.withColumn("dest_severe_weather",   col("dest_severe_weather").cast("int"))

In [ ]:
df.dtypes

[('Year', 'bigint'),
 ('Quarter', 'bigint'),
 ('Month', 'bigint'),
 ('DayofMonth', 'bigint'),
 ('DayOfWeek', 'bigint'),
 ('FlightDate', 'bigint'),
 ('Reporting_Airline', 'string'),
 ('Flight_Number_Reporting_Airline', 'string'),
 ('Origin', 'string'),
 ('Dest', 'string'),
 ('CRSDepTime', 'bigint'),
 ('DepTimeBlk', 'string'),
 ('CRSArrTime', 'bigint'),
 ('ArrDel15', 'bigint'),
 ('CRSElapsedTime', 'double'),
 ('Distance', 'double'),
 ('DistanceGroup', 'bigint'),
 ('date', 'string'),
 ('dep_hour', 'bigint'),
 ('arr_hour', 'bigint'),
 ('dep_hour_minus2', 'bigint'),
 ('arr_hour_minus2', 'bigint'),
 ('origin_temp_f', 'double'),
 ('origin_dewpoint_f', 'double'),
 ('origin_humidity', 'double'),
 ('origin_feels_like_f', 'double'),
 ('origin_wind_kts', 'double'),
 ('origin_gust_kts', 'double'),
 ('origin_visibility', 'double'),
 ('origin_precip_in', 'double'),
 ('origin_wx_codes', 'string'),
 ('origin_is_rain', 'int'),
 ('origin_is_snow', 'int'),
 ('origin_is_fog', 'int'),
 ('origin_low_visibili

# Refreshed Missingness Analysis

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType, FloatType, DoubleType

# ------------------------------------------------------------------
# Cache only if reusing df; otherwise skip to save memory
# df = df.persist()  # optional
# ------------------------------------------------------------------

# Build one aggregation expression per column → single pass over data
null_exprs = []
for field in df.schema.fields:
    c = field.name
    dt = field.dataType

    if isinstance(dt, (FloatType, DoubleType)):
        cond = F.col(c).isNull() | F.isnan(F.col(c))
    elif isinstance(dt, StringType):
        cond = F.col(c).isNull() | (F.trim(F.col(c)) == F.lit(""))
    else:
        cond = F.col(c).isNull()

    null_exprs.append(F.sum(cond.cast("long")).alias(c))

# Add total row count in the SAME aggregation → still one pass
null_exprs.append(F.count(F.lit(1)).alias("__total_rows__"))

# Execute: returns ONE row, very small → safe to collect
agg_row = df.agg(*null_exprs).collect()[0].asDict()
total_rows = agg_row.pop("__total_rows__")

print(f"Total rows: {total_rows:,}")

Total rows: 45,968,068


In [ ]:
schema_map = {f.name: f.dataType.simpleString() for f in df.schema.fields}

rows = [
    (col, schema_map[col], int(missing), int(total_rows),
     round(missing / total_rows * 100, 4))
    for col, missing in agg_row.items()
]

summary_df = (
    spark.createDataFrame(rows, ["column", "dtype", "missing", "total", "missing_pct"])
         .orderBy(F.desc("missing_pct"))
)

summary_df.show(len(rows), truncate=False)

+-------------------------------+------+--------+--------+-----------+
|column                         |dtype |missing |total   |missing_pct|
+-------------------------------+------+--------+--------+-----------+
|dest_wx_codes                  |string|38872269|45968068|84.5636    |
|origin_wx_codes                |string|38614757|45968068|84.0034    |
|origin_gust_kts                |double|34766584|45968068|75.632     |
|dest_gust_kts                  |double|34132915|45968068|74.2535    |
|origin_feels_like_f            |double|1635446 |45968068|3.5578     |
|origin_humidity                |double|1626422 |45968068|3.5382     |
|dest_feels_like_f              |double|1622987 |45968068|3.5307     |
|dest_humidity                  |double|1615268 |45968068|3.5139     |
|origin_dewpoint_f              |double|1592532 |45968068|3.4644     |
|origin_temp_f                  |double|1586034 |45968068|3.4503     |
|dest_dewpoint_f                |double|1584498 |45968068|3.447      |
|dest_

In [ ]:
from pyspark.sql import functions as F

# ------------------------------------------------------------------
# BUCKET A: Weather codes — empty means "no event reported"
# ------------------------------------------------------------------
df = df.withColumn(
    "origin_wx_codes",
    F.when(F.col("origin_wx_codes").isNull() | (F.trim("origin_wx_codes") == ""), "none")
     .otherwise(F.col("origin_wx_codes"))
).withColumn(
    "dest_wx_codes",
    F.when(F.col("dest_wx_codes").isNull() | (F.trim("dest_wx_codes") == ""), "none")
     .otherwise(F.col("dest_wx_codes"))
)

# ------------------------------------------------------------------
# BUCKET B: Gusts — NULL means "no gust beyond steady wind"
# ------------------------------------------------------------------
df = df.fillna({"origin_gust_kts": 0.0, "dest_gust_kts": 0.0})

# ------------------------------------------------------------------
# BUCKET D: ArrDel15 — drop cancelled/diverted flights for modeling
# ------------------------------------------------------------------
df = df.filter(F.col("ArrDel15").isNotNull())

# ------------------------------------------------------------------
# BUCKET E: Trivial nulls — safe to drop
# ------------------------------------------------------------------
df = df.dropna(subset=["CRSElapsedTime", "Flight_Number_Reporting_Airline"])

In [ ]:
df = df.withColumn(
    "origin_weather_missing",
    F.col("origin_temp_f").isNull().cast("int")
).withColumn(
    "dest_weather_missing",
    F.col("dest_temp_f").isNull().cast("int")
)

# Zero-fill numeric weather; tree models will learn from the flag
df = df.fillna(0, subset=weather_cols + [
    "origin_is_rain", "origin_is_snow", "origin_is_fog",
    "origin_low_visibility", "origin_high_wind", "origin_severe_weather",
    "dest_is_rain", "dest_is_snow", "dest_is_fog",
    "dest_low_visibility", "dest_high_wind", "dest_severe_weather",
])

In [ ]:
null_exprs = []
for field in df.schema.fields:
    c, dt = field.name, field.dataType
    if dt.simpleString() in ("double", "float"):
        cond = F.col(c).isNull() | F.isnan(F.col(c))
    elif dt.simpleString() == "string":
        cond = F.col(c).isNull() | (F.trim(F.col(c)) == "")
    else:
        cond = F.col(c).isNull()
    null_exprs.append(F.sum(cond.cast("long")).alias(c))

result = df.agg(*null_exprs).collect()[0].asDict()
remaining = {k: v for k, v in result.items() if v > 0}
print("Remaining missing values:", remaining if remaining else "✅ None!")

Remaining missing values: ✅ None!


# Split and Save Data

In [ ]:
from pyspark.sql import functions as F

(
    df.groupBy("Year")
      .agg(F.count("*").alias("rows"))
      .orderBy("Year")
      .show()
)

+----+-------+
|Year|   rows|
+----+-------+
|2018|7071464|
|2019|7268232|
|2020|4399575|
|2021|5878219|
|2022|6532012|
|2023|6743403|
|2024|6965246|
+----+-------+



In [ ]:
# Time-based split boundaries
TRAIN_YEARS    = [2018, 2019, 2020, 2021, 2022]
VALIDATE_YEARS = [2023]
TEST_YEARS     = [2024]

train_df    = df.filter(F.col("Year").isin(TRAIN_YEARS))
validate_df = df.filter(F.col("Year").isin(VALIDATE_YEARS))
test_df     = df.filter(F.col("Year").isin(TEST_YEARS))

In [ ]:
def summarize(name, d):
    total   = d.count()
    pos     = d.filter(F.col("ArrDel15") == 1).count()
    pos_pct = pos / total * 100 if total else 0
    print(f"{name:10s} | rows: {total:>11,} | delayed: {pos:>10,} ({pos_pct:.2f}%)")

summarize("TRAIN",    train_df)
summarize("VALIDATE", validate_df)
summarize("TEST",     test_df)

TRAIN      | rows:  31,149,502 | delayed:  5,560,469 (17.85%)
VALIDATE   | rows:   6,743,403 | delayed:  1,386,699 (20.56%)
TEST       | rows:   6,965,246 | delayed:  1,449,966 (20.82%)


In [ ]:
import shutil, os, glob
from google.colab import drive
drive.mount('/content/drive')

BASE_LOCAL = "/content/flights_split"
BASE_DRIVE = "/content/drive/MyDrive/flights_split"

splits = {
    "train_2018_2022":    train_df,
    "validate_2023": validate_df,
    "test_2024":     test_df,
}

os.makedirs(BASE_LOCAL, exist_ok=True)

for name, sdf in splits.items():
    print(f"\n→ Writing {name} ...")

    tmp_dir = f"{BASE_LOCAL}/{name}_tmp"
    final_file = f"{BASE_LOCAL}/{name}.parquet"

    # 1. Spark writes to a temp directory with a single part-file
    (
        sdf.coalesce(1)
           .write.mode("overwrite")
           .parquet(f"file://{tmp_dir}")
    )

    # 2. Find the single part-*.parquet file and rename it
    part_file = glob.glob(f"{tmp_dir}/part-*.parquet")[0]
    shutil.move(part_file, final_file)

    # 3. Remove the now-empty temp dir (and its _SUCCESS, .crc files)
    shutil.rmtree(tmp_dir)

    # 4. Report size
    size_gb = os.path.getsize(final_file) / 1024**3
    print(f"   ✅ {name}.parquet written locally ({size_gb:.2f} GB)")

# Bulk-copy all single files to Drive
print("\n→ Copying to Google Drive ...")
os.makedirs(BASE_DRIVE, exist_ok=True)
for name in splits.keys():
    src = f"{BASE_LOCAL}/{name}.parquet"
    dst = f"{BASE_DRIVE}/{name}.parquet"
    shutil.copy2(src, dst)
    print(f"   ✅ Copied {name}.parquet to Drive")

print(f"\n✅ All splits saved as single files to {BASE_DRIVE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

→ Writing train ...
   ✅ train.parquet written locally (1.01 GB)

→ Writing validate ...
   ✅ validate.parquet written locally (0.22 GB)

→ Writing test ...
   ✅ test.parquet written locally (0.22 GB)

→ Copying to Google Drive ...
   ✅ Copied train.parquet to Drive
   ✅ Copied validate.parquet to Drive
   ✅ Copied test.parquet to Drive

✅ All splits saved as single files to /content/drive/MyDrive/flights_split
